# Construcción del dataset de Beyblade X

Este notebook se concentra **solo en recolección, limpieza, unión y preparación del dataset**.  
El PCA se realizará en otro notebook.

Flujo:

1. **Beywatch** → estadísticas competitivas de Blades, Ratchets y Bits.
2. **Rucua** → precio, stock y contenido comercial de los productos.
3. **Normalización** → identificar qué piezas contiene cada producto.
4. **Enriquecimiento** → completar dos sets que Rucua no describe con suficiente detalle.
5. **Cruce Rucua + Beywatch** → llevar el meta competitivo al nivel de producto.
6. **Escenarios de ingreso** → crear las observaciones finales para el notebook de PCA.
7. **Exportación** → guardar datasets intermedios y `beyblade_pca_input.csv`.

> Ejecuta el notebook de arriba hacia abajo. Las celdas están ordenadas para funcionar con **Run All**.

## 0. Dependencias e imports

Los imports están concentrados en una sola sección para evitar errores como `pd is not defined`, `np is not defined` o `date is not defined` después de reiniciar el kernel.

In [1]:
%pip install -q requests beautifulsoup4 pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import time
import unicodedata
from datetime import date
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup

### Configuración general

- `REQUEST_TIMEOUT`: máximo de espera por petición.
- `REQUEST_DELAY`: pausa entre peticiones para no bombardear los sitios.
- `ARCHIVO_SEED`: Excel que contiene los escenarios personales de ingreso.

In [3]:
HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

REQUEST_TIMEOUT = 20
REQUEST_DELAY = 1.0

ARCHIVO_SEED = Path("beyblade_dataset_v1.xlsx")

## 1. Utilidades comunes

Estas funciones se reutilizan en ambas fuentes.

In [4]:
def normalizar_texto(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).replace("\xa0", " ").replace("Â", "")
    return " ".join(texto.strip().split())


def normalizar_nombre(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        c for c in texto
        if not unicodedata.combining(c)
    )

    texto = texto.replace("'", "")
    texto = texto.replace("-", " ")
    texto = " ".join(texto.split())

    return texto


def reintentar_urls(errores, funcion, intentos=2, pausa=2.0):
    """Reintenta solamente URLs que fallaron; evita repetir todo el scraping."""

    recuperados = []
    pendientes = errores.copy()

    for intento in range(1, intentos + 1):
        if not pendientes:
            break

        print(f"Reintento {intento}: {len(pendientes)} URL(s)")
        nuevos_pendientes = []

        for item in pendientes:
            url = item["url"]

            try:
                recuperados.append(funcion(url))
            except Exception as e:
                nuevos_pendientes.append({
                    "url": url,
                    "error": str(e)
                })

            time.sleep(pausa)

        pendientes = nuevos_pendientes

    return recuperados, pendientes

# 2. Beywatch: meta competitivo

Cada pieza se guarda individualmente. Las métricas principales son:

- `first_rate_pct`
- `win_share_pct`
- `usage_pct`
- `top_cuts`

`NaN` **no se convierte en 0**: significa que Beywatch no publicó la métrica, algo posible en piezas nuevas o con información insuficiente.

In [5]:
def detectar_tipo_beywatch(url):
    if "/blades/" in url:
        return "blade"

    if "/parts/ratchet-" in url:
        return "ratchet"

    if "/parts/bit-" in url:
        return "bit"

    return "unknown"


def obtener_estadisticas_beywatch(url):
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")
    texto = soup.get_text(" ", strip=True)

    h1 = soup.find("h1")
    if h1 is None:
        raise ValueError(f"No se encontró <h1> en {url}")

    nombre = normalizar_texto(
        h1.get_text(" ", strip=True)
    )

    def porcentaje(etiqueta):
        match = re.search(
            rf"{re.escape(etiqueta)}.*?([\d.]+)%",
            texto,
            re.IGNORECASE
        )
        return float(match.group(1)) if match else None

    match_cuts = re.search(
        r"Top Cuts.*?([\d,]+)",
        texto,
        re.IGNORECASE
    )

    top_cuts = (
        int(match_cuts.group(1).replace(",", ""))
        if match_cuts
        else None
    )

    return {
        "name": nombre,
        "part_type": detectar_tipo_beywatch(url),
        "first_rate_pct": porcentaje("1st Place Rate"),
        "win_share_pct": porcentaje("Win Share"),
        "usage_pct": porcentaje("Usage"),
        "top_cuts": top_cuts,
        "snapshot_date": date.today().isoformat(),
        "source_url": url
    }

### 2.1 Descubrimiento automático de URLs

Se consultan los catálogos de Blade, Ratchet y Bit para no escribir manualmente cientos de enlaces.

In [6]:
PAGINAS_CATALOGO_BEYWATCH = [
    ("blade", "https://beywatch.gg/"),
    ("ratchet", "https://beywatch.gg/parts/ratchets"),
    ("bit", "https://beywatch.gg/parts/bits")
]

links_beywatch = []

for tipo, url_catalogo in PAGINAS_CATALOGO_BEYWATCH:
    response = requests.get(
        url_catalogo,
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    for etiqueta in soup.find_all("a", href=True):
        href = etiqueta["href"]

        if tipo == "blade" and href.startswith("/blades/"):
            links_beywatch.append(href)

        elif tipo == "ratchet" and href.startswith("/parts/ratchet-"):
            links_beywatch.append(href)

        elif tipo == "bit" and href.startswith("/parts/bit-"):
            links_beywatch.append(href)

links_beywatch = sorted(set(links_beywatch))

urls_beywatch = [
    urljoin("https://beywatch.gg", link)
    for link in links_beywatch
]

print("Total:", len(urls_beywatch))
print("Blades:", sum("/blades/" in u for u in urls_beywatch))
print("Ratchets:", sum("/parts/ratchet-" in u for u in urls_beywatch))
print("Bits:", sum("/parts/bit-" in u for u in urls_beywatch))

Total: 226
Blades: 135
Ratchets: 37
Bits: 54


### 2.2 Scraping completo y reintentos

In [7]:
resultados_beywatch = []
errores_beywatch = []

for i, url in enumerate(urls_beywatch, start=1):
    print(f"[{i}/{len(urls_beywatch)}] {url}")

    try:
        resultados_beywatch.append(
            obtener_estadisticas_beywatch(url)
        )
    except Exception as e:
        errores_beywatch.append({
            "url": url,
            "error": str(e)
        })

    time.sleep(REQUEST_DELAY)

df_meta_raw = pd.DataFrame(resultados_beywatch)

recuperados, errores_beywatch_finales = reintentar_urls(
    errores_beywatch,
    obtener_estadisticas_beywatch
)

if recuperados:
    df_meta_raw = pd.concat(
        [
            df_meta_raw,
            pd.DataFrame(recuperados)
        ],
        ignore_index=True
    )

df_meta_raw = (
    df_meta_raw
    .drop_duplicates(subset=["source_url"], keep="last")
    .reset_index(drop=True)
)

print("\nDimensiones:", df_meta_raw.shape)
print("Errores restantes:", len(errores_beywatch_finales))

[1/226] https://beywatch.gg/blades/aero-pegasus
[2/226] https://beywatch.gg/blades/antler
[3/226] https://beywatch.gg/blades/arc
[4/226] https://beywatch.gg/blades/bite-croc
[5/226] https://beywatch.gg/blades/black-shell
[6/226] https://beywatch.gg/blades/blast
[7/226] https://beywatch.gg/blades/blitz
[8/226] https://beywatch.gg/blades/brave
[9/226] https://beywatch.gg/blades/brush
[10/226] https://beywatch.gg/blades/bullet-griffon
[11/226] https://beywatch.gg/blades/bumblebee
[12/226] https://beywatch.gg/blades/captain-america
[13/226] https://beywatch.gg/blades/chewbacca
[14/226] https://beywatch.gg/blades/clamp-crab
[15/226] https://beywatch.gg/blades/clock-mirage
[16/226] https://beywatch.gg/blades/cobalt-dragoon
[17/226] https://beywatch.gg/blades/cobalt-drake
[18/226] https://beywatch.gg/blades/crimson-garuda
[19/226] https://beywatch.gg/blades/cutter-shinobi
[20/226] https://beywatch.gg/blades/dark
[21/226] https://beywatch.gg/blades/darth-vader
[22/226] https://beywatch.gg/blad

### 2.3 Limpieza y clave de unión

Para los Bits, `lookup_key` usa la abreviatura entre paréntesis (`Low Rush (LR)` → `LR`).  
Para Blade y Ratchet conserva el nombre.

In [8]:
df_meta_clean = df_meta_raw.copy()

df_meta_clean["has_meta_data"] = (
    df_meta_clean["usage_pct"]
    .notna()
    .astype(int)
)

df_meta_clean["meta_status"] = np.where(
    df_meta_clean["has_meta_data"].eq(1),
    "observed",
    "pending_or_insufficient"
)


def crear_lookup_key(row):
    nombre = normalizar_texto(row["name"])

    if row["part_type"] == "bit":
        match = re.search(r"\(([^)]+)\)", nombre)

        if match:
            return normalizar_texto(
                match.group(1)
            )

    return nombre


df_meta_clean["lookup_key"] = (
    df_meta_clean.apply(
        crear_lookup_key,
        axis=1
    )
)

print(df_meta_clean["part_type"].value_counts())
print("\nMeta disponible:")
print(df_meta_clean["meta_status"].value_counts())
print("\nNulos:")
print(df_meta_clean.isnull().sum())

part_type
blade      135
bit         54
ratchet     37
Name: count, dtype: int64

Meta disponible:
meta_status
observed                   207
pending_or_insufficient     19
Name: count, dtype: int64

Nulos:
name               0
part_type          0
first_rate_pct    19
win_share_pct     19
usage_pct         19
top_cuts          19
snapshot_date      0
source_url         0
has_meta_data      0
meta_status        0
lookup_key         0
dtype: int64


In [9]:
df_meta_raw.to_csv(
    "beywatch_meta_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

df_meta_clean.to_csv(
    "beywatch_meta_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

if errores_beywatch_finales:
    pd.DataFrame(
        errores_beywatch_finales
    ).to_csv(
        "beywatch_scraping_errors.csv",
        index=False,
        encoding="utf-8-sig"
    )

# 3. Rucua: precio, stock y contenido comercial

Rucua publica `JSON-LD` de tipo `Product`.  
El JSON-LD se usa para nombre, precio y stock; la descripción se usa para contar Blades, Ratchets, Bits, lanzadores y estadios.

In [10]:
URL_RUCUA_CATALOGO = (
    "https://rucua.com/collections/vendors?q=Takara+Tomy"
)

response = requests.get(
    URL_RUCUA_CATALOGO,
    headers=HEADERS,
    timeout=REQUEST_TIMEOUT
)
response.raise_for_status()
response.encoding = "utf-8"

soup = BeautifulSoup(
    response.text,
    "html.parser"
)

links_rucua = sorted({
    a["href"]
    for a in soup.find_all("a", href=True)
    if a["href"].startswith("/products/")
})

urls_rucua = [
    urljoin("https://rucua.com", link)
    for link in links_rucua
]

print("Productos encontrados:", len(urls_rucua))

Productos encontrados: 50


### 3.1 Parsers de Rucua

In [11]:
def extraer_cantidad_patron(texto, patron):
    match = re.search(
        rf"{patron}[^,;]*?\((\d+)\)",
        texto,
        re.IGNORECASE
    )

    return int(match.group(1)) if match else 0


def extraer_contenido_rucua(texto):
    texto = normalizar_texto(texto) or ""

    blade_count = extraer_cantidad_patron(
        texto,
        r"\bblade\b"
    )

    ratchet_count = extraer_cantidad_patron(
        texto,
        r"\bratchet\b"
    )

    bit_count = extraer_cantidad_patron(
        texto,
        r"\bbit\b"
    )

    launcher_count = extraer_cantidad_patron(
        texto,
        r"(?:lanzador|launcher)"
    )

    stadium_count = extraer_cantidad_patron(
        texto,
        r"(?:estadio|stadium)"
    )

    bey_count = min(
        blade_count,
        ratchet_count,
        bit_count
    )

    return {
        "blade_count": blade_count,
        "ratchet_count": ratchet_count,
        "bit_count": bit_count,
        "launcher_count": launcher_count,
        "stadium_count": stadium_count,
        "bey_count": bey_count,
        "launcher_included": int(launcher_count > 0),
        "stadium_included": int(stadium_count > 0)
    }


def detectar_tipo_producto(nombre):
    nombre = (nombre or "").lower()

    if "stadium set" in nombre:
        return "Stadium Set"
    if "deck set" in nombre:
        return "Deck Set"
    if "bit set" in nombre:
        return "Bit Set"
    if "ratchet set" in nombre:
        return "Ratchet Set"
    if "blade set" in nombre:
        return "Blade Set"
    if "starter" in nombre:
        return "Starter"
    if "booster" in nombre:
        return "Booster"
    if "launcher" in nombre or "lanzador" in nombre:
        return "Launcher"
    if "set" in nombre:
        return "Set"

    return "Other"

In [12]:
def encontrar_product_jsonld(soup):
    def recorrer(obj):
        if isinstance(obj, dict):
            if obj.get("@type") == "Product":
                return obj

            if "@graph" in obj:
                encontrado = recorrer(obj["@graph"])
                if encontrado:
                    return encontrado

        elif isinstance(obj, list):
            for item in obj:
                encontrado = recorrer(item)
                if encontrado:
                    return encontrado

        return None

    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):
        try:
            data = json.loads(
                script.get_text(strip=True)
            )
        except (json.JSONDecodeError, TypeError):
            continue

        encontrado = recorrer(data)

        if encontrado:
            return encontrado

    return None


def obtener_producto_rucua(url):
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=REQUEST_TIMEOUT
    )
    response.raise_for_status()
    response.encoding = "utf-8"

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    producto_json = encontrar_product_jsonld(soup)

    if producto_json is None:
        raise ValueError(
            "No se encontró Product JSON-LD"
        )

    offers = producto_json.get(
        "offers",
        {}
    )

    if isinstance(offers, list):
        offers = offers[0] if offers else {}

    marca = producto_json.get(
        "brand",
        {}
    )

    disponibilidad = offers.get(
        "availability"
    )

    if disponibilidad:
        disponibilidad = (
            disponibilidad
            .rstrip("/")
            .split("/")[-1]
        )

    descripcion_raw = producto_json.get(
        "description",
        ""
    )

    descripcion_limpia = BeautifulSoup(
        descripcion_raw,
        "html.parser"
    ).get_text(
        " ",
        strip=True
    )

    contenido = extraer_contenido_rucua(
        descripcion_limpia
    )

    precio = offers.get("price")

    try:
        precio = (
            float(precio)
            if precio is not None
            else None
        )
    except (TypeError, ValueError):
        precio = None

    nombre = producto_json.get("name")

    return {
        "source": "Rucua",
        "product_name": nombre,
        "product_type": detectar_tipo_producto(nombre),
        "brand": (
            marca.get("name")
            if isinstance(marca, dict)
            else marca
        ),
        "price": precio,
        "currency": offers.get("priceCurrency"),
        "stock_status": disponibilidad,
        "blade_count": contenido["blade_count"],
        "ratchet_count": contenido["ratchet_count"],
        "bit_count": contenido["bit_count"],
        "launcher_count": contenido["launcher_count"],
        "stadium_count": contenido["stadium_count"],
        "bey_count": contenido["bey_count"],
        "launcher_included": contenido["launcher_included"],
        "stadium_included": contenido["stadium_included"],
        "description_text": descripcion_limpia,
        "listing_url": url,
        "snapshot_date": date.today().isoformat()
    }

### 3.2 Scraping completo de Rucua

In [13]:
resultados_rucua = []
errores_rucua = []

for i, url in enumerate(urls_rucua, start=1):
    print(f"[{i}/{len(urls_rucua)}] {url}")

    try:
        resultados_rucua.append(
            obtener_producto_rucua(url)
        )
    except Exception as e:
        errores_rucua.append({
            "url": url,
            "error": str(e)
        })

    time.sleep(REQUEST_DELAY)

df_rucua_raw = pd.DataFrame(
    resultados_rucua
)

recuperados, errores_rucua_finales = reintentar_urls(
    errores_rucua,
    obtener_producto_rucua
)

if recuperados:
    df_rucua_raw = pd.concat(
        [
            df_rucua_raw,
            pd.DataFrame(recuperados)
        ],
        ignore_index=True
    )

df_rucua_raw = (
    df_rucua_raw
    .drop_duplicates(
        subset=["listing_url"],
        keep="last"
    )
    .reset_index(drop=True)
)

print("\nDimensiones:", df_rucua_raw.shape)
print("Errores restantes:", len(errores_rucua_finales))

[1/50] https://rucua.com/products/bx-00-beyblade-25th-anniversary-set
[2/50] https://rucua.com/products/bx-00-bit-set-f-t-b-n-gold-x-black
[3/50] https://rucua.com/products/bx-00-bit-set-f-t-b-n-silver-x-white
[4/50] https://rucua.com/products/bx-00-booster-cobalt-drake-4-60f-clear-ver
[5/50] https://rucua.com/products/bx-00-booster-cobalt-drake-4-60f-metal-coat-blue-ver
[6/50] https://rucua.com/products/bx-00-booster-draciel-shield-7-60d
[7/50] https://rucua.com/products/bx-00-booster-dragoon-storm-4-60ra
[8/50] https://rucua.com/products/bx-00-booster-dran-sword-3-60f-version-2-0
[9/50] https://rucua.com/products/bx-00-booster-dranzer-spiral-3-80t
[10/50] https://rucua.com/products/bx-00-booster-dranzer-spiral-3-80t-black-ver
[11/50] https://rucua.com/products/bx-00-booster-drigger-slash-4-80p
[12/50] https://rucua.com/products/bx-00-booster-hells-chain-5-60ht-metal-coat-black
[13/50] https://rucua.com/products/bx-00-booster-hells-size-4-60t-metal-coat-gold
[14/50] https://rucua.com/

### 3.3 Auditoría RAW de Rucua

In [14]:
print("Tipos de producto:")
print(df_rucua_raw["product_type"].value_counts(dropna=False))

print("\nStock:")
print(df_rucua_raw["stock_status"].value_counts(dropna=False))

print("\nCantidad de Beys:")
print(df_rucua_raw["bey_count"].value_counts(dropna=False))

print("\nNulos:")
print(df_rucua_raw.isnull().sum())

Tipos de producto:
product_type
Booster     18
Starter     12
Other       11
Set          2
Bit Set      2
Launcher     2
Deck Set     1
Name: count, dtype: int64

Stock:
stock_status
InStock       28
OutOfStock    20
Name: count, dtype: int64

Cantidad de Beys:
bey_count
1    35
0    11
4     1
3     1
Name: count, dtype: int64

Nulos:
source               0
product_name         0
product_type         0
brand                0
price                0
currency             0
stock_status         0
blade_count          0
ratchet_count        0
bit_count            0
launcher_count       0
stadium_count        0
bey_count            0
launcher_included    0
stadium_included     0
description_text     0
listing_url          0
snapshot_date        0
dtype: int64


In [15]:
df_rucua_raw.to_csv(
    "rucua_products_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

if errores_rucua_finales:
    pd.DataFrame(
        errores_rucua_finales
    ).to_csv(
        "rucua_scraping_errors.csv",
        index=False,
        encoding="utf-8-sig"
    )

# 4. Normalización de productos Rucua

Se crean identificadores y se extrae del nombre comercial:

- código `BX / UX / CX`;
- Ratchet;
- Bit;
- Blade.

Para Blade usamos los nombres reales de Beywatch como diccionario.  
Los alias permiten resolver diferencias de escritura entre tiendas.

In [16]:
df_rucua_clean = df_rucua_raw.copy()

df_rucua_clean["listing_id"] = (
    "rucua_"
    + df_rucua_clean["listing_url"]
        .str.rstrip("/")
        .str.split("/")
        .str[-1]
)


def extraer_product_code(nombre):
    match = re.search(
        r"\b(BX|UX|CX)-\d{2}\b",
        str(nombre),
        re.IGNORECASE
    )

    return (
        match.group(0).upper()
        if match
        else None
    )


def extraer_combo_stock(nombre):
    match = re.search(
        r"\b(\d-\d{2})([A-Z]{1,3})\b",
        str(nombre)
    )

    if match:
        return {
            "ratchet": match.group(1),
            "bit": match.group(2)
        }

    return {
        "ratchet": None,
        "bit": None
    }


df_rucua_clean["product_code"] = (
    df_rucua_clean["product_name"]
    .apply(extraer_product_code)
)

combos = (
    df_rucua_clean["product_name"]
    .apply(extraer_combo_stock)
    .apply(pd.Series)
)

df_rucua_clean[
    ["ratchet", "bit"]
] = combos[
    ["ratchet", "bit"]
]

In [17]:
blades_conocidos = (
    df_meta_clean[
        df_meta_clean["part_type"].eq("blade")
    ]["lookup_key"]
    .dropna()
    .unique()
    .tolist()
)

blade_aliases = {
    "drigger slash": "Driger Slash",
    "mammoth task": "Tusk Mammoth",
    "mammoth tusk": "Tusk Mammoth",
    "croc crunch": "Bite Croc",
    "croco crunch": "Bite Croc",
    "storm pegasus": "Storm Pegasis",
    "xeno excalibur": "Xeno Xcalibur"
}


def detectar_blade(nombre, blades):
    nombre_norm = normalizar_nombre(nombre)

    for alias, blade_real in blade_aliases.items():
        if normalizar_nombre(alias) in nombre_norm:
            return blade_real

    coincidencias = []

    for blade in blades:
        blade_norm = normalizar_nombre(blade)

        if blade_norm and blade_norm in nombre_norm:
            coincidencias.append(blade)

    if not coincidencias:
        return None

    return max(
        coincidencias,
        key=len
    )


df_rucua_clean["blade"] = (
    df_rucua_clean["product_name"]
    .apply(
        lambda nombre: detectar_blade(
            nombre,
            blades_conocidos
        )
    )
)

### 4.1 Estado del matching de Blade

No todos los `None` son errores:

- un launcher o estadio no debe tener Blade;
- un set con varios Beys necesita más de una fila de composición;
- Lightning L-Drago se mantiene como Blade ambiguo porque Beywatch distingue más de una variante.

In [18]:
def clasificar_match_blade(row):
    if row["blade_count"] == 0:
        return "no_blade_expected"

    if row["blade_count"] > 1:
        return "multi_bey_set"

    nombre = normalizar_nombre(
        row["product_name"]
    )

    if "lightning l drago" in nombre:
        return "ambiguous_blade"

    if pd.notna(row["blade"]):
        return "matched"

    return "needs_review"


df_rucua_clean["blade_match_status"] = (
    df_rucua_clean.apply(
        clasificar_match_blade,
        axis=1
    )
)

print(
    df_rucua_clean[
        "blade_match_status"
    ].value_counts()
)

blade_match_status
matched              36
no_blade_expected     8
multi_bey_set         2
ambiguous_blade       1
needs_review          1
Name: count, dtype: int64


# 5. Composición de paquetes

`df_rucua_clean` tiene una fila por publicación comercial.  
`df_package_contents` tendrá una fila por Bey incluido en esa publicación.

Esto evita columnas artificiales como `blade_1`, `blade_2`, `blade_3` y permite trabajar con paquetes de cualquier tamaño.

In [19]:
alias_a_blade = {}

for blade in blades_conocidos:
    alias_a_blade[
        normalizar_nombre(blade)
    ] = blade

for alias, blade_real in blade_aliases.items():
    alias_a_blade[
        normalizar_nombre(alias)
    ] = blade_real


def extraer_beys_del_texto(texto):
    if pd.isna(texto):
        return []

    texto = str(texto)

    patron_combo = re.compile(
        r"\b(\d-\d{2})([A-Z]{1,3})\b"
    )

    resultados = []

    for match in patron_combo.finditer(texto):
        ratchet = match.group(1)
        bit = match.group(2)

        inicio = max(
            0,
            match.start() - 100
        )

        prefijo = texto[
            inicio:match.start()
        ]

        prefijo_norm = normalizar_nombre(
            prefijo
        )

        candidatos = []

        for alias_norm, blade_real in alias_a_blade.items():
            coincidencias = list(
                re.finditer(
                    rf"\b{re.escape(alias_norm)}\b",
                    prefijo_norm
                )
            )

            if coincidencias:
                ultima = coincidencias[-1]

                candidatos.append(
                    (
                        ultima.start(),
                        len(alias_norm),
                        blade_real
                    )
                )

        blade = None

        if candidatos:
            candidatos.sort(
                reverse=True
            )
            blade = candidatos[0][2]

        resultados.append({
            "blade": blade,
            "ratchet": ratchet,
            "bit": bit
        })

    unicos = []
    vistos = set()

    for item in resultados:
        clave = (
            item["blade"],
            item["ratchet"],
            item["bit"]
        )

        if clave not in vistos:
            vistos.add(clave)
            unicos.append(item)

    return unicos

In [20]:
filas_contenido = []

for _, fila in df_rucua_clean.iterrows():
    listing_id = fila["listing_id"]
    blade_count = fila["blade_count"]

    # Accesorios y sets de piezas sin Bey completo.
    if blade_count == 0:
        continue

    # Producto de un solo Bey:
    # conserva lo que haya sido posible resolver desde el título.
    if blade_count == 1:
        if (
            pd.notna(fila["blade"])
            or pd.notna(fila["ratchet"])
            or pd.notna(fila["bit"])
        ):
            filas_contenido.append({
                "listing_id": listing_id,
                "slot": 1,
                "blade": fila["blade"],
                "ratchet": fila["ratchet"],
                "bit": fila["bit"],
                "content_source": "product_name",
                "source_reference": fila["listing_url"]
            })
            continue

    # Sets con varios Beys o títulos que no identifican el combo.
    beys = extraer_beys_del_texto(
        fila["description_text"]
    )

    for slot, bey in enumerate(
        beys,
        start=1
    ):
        filas_contenido.append({
            "listing_id": listing_id,
            "slot": slot,
            "blade": bey["blade"],
            "ratchet": bey["ratchet"],
            "bit": bey["bit"],
            "content_source": "description",
            "source_reference": fila["listing_url"]
        })

df_package_contents = pd.DataFrame(
    filas_contenido
)

### 5.1 Enriquecimiento externo de BX-07 y BX-08

Rucua indica cuántos Beys traen estos sets, pero no enumera sus combos en la descripción.  
Se agregan manualmente con fuentes de producto/manual de Takara Tomy para conservar trazabilidad.

Este enriquecimiento está separado del scraping para que quede claro qué filas no provienen de Rucua.

In [21]:
composiciones_externas = [
    {
        "listing_id": "rucua_bx-07-start-dash-set",
        "slot": 1,
        "blade": "Dran Sword",
        "ratchet": "3-60",
        "bit": "F",
        "content_source": "takara_tomy",
        "source_reference": (
            "https://beyblade.takaratomy.co.jp/"
            "beyblade-x/lineup/bx07.html"
        )
    },
    {
        "listing_id": "rucua_bx-08-3on3-deck-set",
        "slot": 1,
        "blade": "Hells Scythe",
        "ratchet": "3-80",
        "bit": "B",
        "content_source": "takara_tomy",
        "source_reference": (
            "https://beyblade.takaratomy.co.jp/"
            "beyblade-x/lineup/bx08.html"
        )
    },
    {
        "listing_id": "rucua_bx-08-3on3-deck-set",
        "slot": 2,
        "blade": "Wizard Arrow",
        "ratchet": "4-60",
        "bit": "N",
        "content_source": "takara_tomy",
        "source_reference": (
            "https://beyblade.takaratomy.co.jp/"
            "beyblade-x/lineup/bx08.html"
        )
    },
    {
        "listing_id": "rucua_bx-08-3on3-deck-set",
        "slot": 3,
        "blade": "Knight Shield",
        "ratchet": "4-80",
        "bit": "T",
        "content_source": "takara_tomy",
        "source_reference": (
            "https://beyblade.takaratomy.co.jp/"
            "beyblade-x/lineup/bx08.html"
        )
    }
]

df_enrichment = pd.DataFrame(
    composiciones_externas
)

# Para estos dos sets la fuente oficial tiene prioridad.
# Si Rucua llegara a producir una extracción parcial, se reemplaza
# por la composición externa completa para evitar duplicados o mezclas.
ids_enriquecidos = set(
    df_enrichment["listing_id"]
)

df_package_contents = df_package_contents[
    ~df_package_contents["listing_id"].isin(
        ids_enriquecidos
    )
].copy()

df_package_contents = pd.concat(
    [
        df_package_contents,
        df_enrichment
    ],
    ignore_index=True
)

### 5.2 Auditoría de composición

Se compara el número de Beys declarado por Rucua con el número de filas de composición obtenidas.

Una auditoría vacía significa que todos los productos con Beys tienen el número esperado de registros.

In [22]:
def clasificar_composicion(row):
    if row["blade_count"] == 0:
        return "not_applicable"

    if row["blade_count"] > 1:
        return "multi_bey"

    if row["blade_match_status"] == "ambiguous_blade":
        return "ambiguous"

    if pd.notna(row["blade"]):
        return "resolved"

    return "unresolved"


df_rucua_clean["composition_status"] = (
    df_rucua_clean.apply(
        clasificar_composicion,
        axis=1
    )
)

conteo_extraido = (
    df_package_contents
    .groupby("listing_id")
    .size()
    .reset_index(
        name="beys_extraidos"
    )
)

auditoria_contenido = (
    df_rucua_clean[
        [
            "listing_id",
            "product_name",
            "blade_count",
            "composition_status"
        ]
    ]
    .merge(
        conteo_extraido,
        how="left",
        on="listing_id"
    )
)

auditoria_contenido["beys_extraidos"] = (
    auditoria_contenido["beys_extraidos"]
    .fillna(0)
    .astype(int)
)

auditoria_contenido["contenido_completo"] = (
    auditoria_contenido["blade_count"]
    ==
    auditoria_contenido["beys_extraidos"]
)

problemas_composicion = auditoria_contenido[
    ~auditoria_contenido["contenido_completo"]
]

display(problemas_composicion)

,listing_id,product_name,blade_count,composition_status,beys_extraidos,contenido_completo


In [23]:
df_package_contents["blade_resuelto"] = (
    df_package_contents["blade"].notna()
)

df_package_contents["ratchet_resuelto"] = (
    df_package_contents["ratchet"].notna()
)

df_package_contents["bit_resuelto"] = (
    df_package_contents["bit"].notna()
)

df_package_contents["combo_resuelto"] = (
    df_package_contents["blade_resuelto"]
    & df_package_contents["ratchet_resuelto"]
    & df_package_contents["bit_resuelto"]
)

print(
    df_package_contents[
        "combo_resuelto"
    ].value_counts(dropna=False)
)

combo_resuelto
True     40
False     5
Name: count, dtype: int64


In [24]:
df_rucua_clean.to_csv(
    "rucua_products_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

df_package_contents.to_csv(
    "rucua_package_contents.csv",
    index=False,
    encoding="utf-8-sig"
)

# 6. Cruzar composición Rucua con meta de Beywatch

Cada fila de `df_package_contents` representa un Bey dentro de un producto.  
Ahora se añaden las métricas de su Blade, Ratchet y Bit.

In [25]:
def preparar_meta_para_merge(
    df_meta,
    part_type,
    key_name,
    prefix
):
    tabla = df_meta[
        df_meta["part_type"].eq(part_type)
    ][
        [
            "lookup_key",
            "first_rate_pct",
            "win_share_pct",
            "usage_pct",
            "top_cuts",
            "has_meta_data"
        ]
    ].copy()

    tabla = tabla.rename(
        columns={
            "lookup_key": key_name,
            "first_rate_pct": f"{prefix}_first_rate_pct",
            "win_share_pct": f"{prefix}_win_share_pct",
            "usage_pct": f"{prefix}_usage_pct",
            "top_cuts": f"{prefix}_top_cuts",
            "has_meta_data": f"{prefix}_has_meta"
        }
    )

    return tabla


meta_blades = preparar_meta_para_merge(
    df_meta_clean,
    "blade",
    "blade",
    "blade"
)

meta_ratchets = preparar_meta_para_merge(
    df_meta_clean,
    "ratchet",
    "ratchet",
    "ratchet"
)

meta_bits = preparar_meta_para_merge(
    df_meta_clean,
    "bit",
    "bit",
    "bit"
)

In [26]:
df_contents_meta = (
    df_package_contents
    .merge(
        meta_blades,
        how="left",
        on="blade"
    )
    .merge(
        meta_ratchets,
        how="left",
        on="ratchet"
    )
    .merge(
        meta_bits,
        how="left",
        on="bit"
    )
)

# Distingue "pieza sin match" de "pieza con match pero sin meta".
for prefijo in ["blade", "ratchet", "bit"]:
    df_contents_meta[
        f"{prefijo}_matched"
    ] = (
        df_contents_meta[
            f"{prefijo}_has_meta"
        ]
        .notna()
        .astype(int)
    )

df_contents_meta["combo_match_coverage"] = (
    df_contents_meta[
        [
            "blade_matched",
            "ratchet_matched",
            "bit_matched"
        ]
    ].mean(axis=1)
)

df_contents_meta["combo_meta_coverage"] = (
    df_contents_meta[
        [
            "blade_has_meta",
            "ratchet_has_meta",
            "bit_has_meta"
        ]
    ]
    .fillna(0)
    .mean(axis=1)
)

### 6.1 Features competitivas por Bey

No se crea una calificación subjetiva.  
Se conservan resúmenes descriptivos:

- media de las piezas;
- máximo de las piezas.

El máximo captura el caso "el paquete vale la pena por una sola pieza fuerte".

In [27]:
def agregar_resumen_combo(
    df,
    columnas,
    nombre
):
    df[f"combo_{nombre}_mean"] = (
        df[columnas]
        .mean(axis=1, skipna=True)
    )

    df[f"combo_{nombre}_max"] = (
        df[columnas]
        .max(axis=1, skipna=True)
    )


agregar_resumen_combo(
    df_contents_meta,
    [
        "blade_usage_pct",
        "ratchet_usage_pct",
        "bit_usage_pct"
    ],
    "usage"
)

agregar_resumen_combo(
    df_contents_meta,
    [
        "blade_top_cuts",
        "ratchet_top_cuts",
        "bit_top_cuts"
    ],
    "top_cuts"
)

agregar_resumen_combo(
    df_contents_meta,
    [
        "blade_first_rate_pct",
        "ratchet_first_rate_pct",
        "bit_first_rate_pct"
    ],
    "first_rate"
)

agregar_resumen_combo(
    df_contents_meta,
    [
        "blade_win_share_pct",
        "ratchet_win_share_pct",
        "bit_win_share_pct"
    ],
    "win_share"
)

### 6.2 Agregación al nivel del producto

Después de analizar cada Bey del paquete, regresamos a una fila por publicación comercial.

In [28]:
meta_por_producto = (
    df_contents_meta
    .groupby("listing_id")
    .agg(
        package_bey_count=(
            "slot",
            "count"
        ),

        package_usage_mean=(
            "combo_usage_mean",
            "mean"
        ),
        package_usage_max=(
            "combo_usage_max",
            "max"
        ),

        package_top_cuts_mean=(
            "combo_top_cuts_mean",
            "mean"
        ),
        package_top_cuts_max=(
            "combo_top_cuts_max",
            "max"
        ),

        package_first_rate_mean=(
            "combo_first_rate_mean",
            "mean"
        ),
        package_first_rate_max=(
            "combo_first_rate_max",
            "max"
        ),

        package_win_share_mean=(
            "combo_win_share_mean",
            "mean"
        ),
        package_win_share_max=(
            "combo_win_share_max",
            "max"
        ),

        package_match_coverage=(
            "combo_match_coverage",
            "mean"
        ),
        package_meta_coverage=(
            "combo_meta_coverage",
            "mean"
        )
    )
    .reset_index()
)

df_rucua_modelo = (
    df_rucua_clean
    .merge(
        meta_por_producto,
        how="left",
        on="listing_id"
    )
)

df_rucua_modelo["in_stock"] = (
    df_rucua_modelo["stock_status"]
    .eq("InStock")
    .astype(int)
)

display(
    df_rucua_modelo[
        [
            "product_name",
            "product_type",
            "price",
            "bey_count",
            "launcher_included",
            "stadium_included",
            "package_usage_mean",
            "package_usage_max",
            "package_meta_coverage"
        ]
    ].head(20)
)

,product_name,product_type,price,bey_count,launcher_included,stadium_included,package_usage_mean,package_usage_max,package_meta_coverage
0,BX-00 Beyblade 25th Anniversary Set,Set,5199.0,4,1,0,5.550000,46.5,1.000000
1,BX-00 Bit Set F/T/B/N Gold x Black,Bit Set,499.0,0,0,0,NaN,NaN,NaN
2,BX-00 Bit Set F/T/B/N Silver x White,Bit Set,499.0,0,0,0,NaN,NaN,NaN
3,BX-00 Booster Cobalt Drake 4-60F Clear Ver.,Booster,1599.0,1,0,0,2.333333,3.2,1.000000
4,BX-00 Booster Cobalt Drake 4-60F Metal Coat: B...,Booster,4699.0,1,0,0,2.333333,3.2,1.000000
5,BX-00 Booster Draciel Shield 7-60D,Booster,599.0,1,0,0,8.133333,23.7,1.000000
6,BX-00 Booster Dragoon Storm 4-60RA,Booster,599.0,1,0,0,1.933333,2.8,1.000000
7,BX-00 Booster Dran Sword 3-60F Version 2.0,Booster,999.0,1,0,0,17.166667,46.5,1.000000
8,BX-00 Booster Dranzer Spiral 3-80T,Booster,599.0,1,0,0,2.666667,6.2,1.000000
9,BX-00 Booster Dranzer Spiral 3-80T Black Ver.,Booster,699.0,1,0,0,2.666667,6.2,1.000000


In [29]:
df_contents_meta.to_csv(
    "rucua_contents_with_meta.csv",
    index=False,
    encoding="utf-8-sig"
)

df_rucua_modelo.to_csv(
    "rucua_model_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

# 7. Preparar el dataset que recibirá el notebook de PCA

No se ejecuta PCA aquí.

Se crean escenarios `producto × ingreso mensual` y dos variables económicas:

- `price_income_pct`: porcentaje del ingreso mensual que representa la compra.
- `price_per_bey`: precio dividido entre la cantidad de Beys completos del paquete.

Los accesorios con `bey_count = 0` se excluyen de este dataset porque el objetivo actual es estudiar la compra de productos que contienen al menos un Bey.

In [30]:
def cargar_ingresos_personales(path_excel):
    if not path_excel.exists():
        raise FileNotFoundError(
            f"No se encontró {path_excel}. "
            "Coloca el Excel junto al notebook."
        )

    xls = pd.ExcelFile(path_excel)

    candidatos = [
        "Income_Scenarios_4k_9k",
        "Ingresos_personales",
        "Ingresos Personales"
    ]

    for sheet in candidatos:
        if sheet in xls.sheet_names:
            df = pd.read_excel(
                path_excel,
                sheet_name=sheet
            )

            if (
                "ingreso_personal_mensual_mxn"
                in df.columns
            ):
                return df

    # Fallback: busca la columna en cualquier hoja.
    for sheet in xls.sheet_names:
        df = pd.read_excel(
            path_excel,
            sheet_name=sheet
        )

        if (
            "ingreso_personal_mensual_mxn"
            in df.columns
        ):
            print(
                f"Hoja de ingresos detectada: {sheet}"
            )
            return df

    raise ValueError(
        "No se encontró una hoja con la columna "
        "'ingreso_personal_mensual_mxn'."
    )


ingresos_personales = cargar_ingresos_personales(
    ARCHIVO_SEED
)

columnas_ingreso = [
    "ingreso_personal_mensual_mxn"
]

if "scenario_id" in ingresos_personales.columns:
    columnas_ingreso.insert(
        0,
        "scenario_id"
    )
else:
    ingresos_personales = (
        ingresos_personales
        .reset_index(drop=True)
    )
    ingresos_personales[
        "scenario_id"
    ] = (
        "SCENARIO_"
        + (
            ingresos_personales.index + 1
        ).astype(str)
    )
    columnas_ingreso.insert(
        0,
        "scenario_id"
    )

ingresos_modelo = (
    ingresos_personales[
        columnas_ingreso
    ]
    .dropna(
        subset=[
            "ingreso_personal_mensual_mxn"
        ]
    )
    .drop_duplicates()
    .copy()
)

display(ingresos_modelo)

,scenario_id,ingreso_personal_mensual_mxn
0,LOW_01,4001
1,LOW_02,4500
2,LOW_03,5000
3,LOW_04,5500
4,LOW_05,6000
5,LOW_06,6500
6,LOW_07,7000
7,LOW_08,7500
8,LOW_09,8000
9,LOW_10,8500


In [31]:
# Base de productos para PCA:
# - contiene al menos un Bey;
# - tiene precio válido en MXN.

df_pca_productos = df_rucua_modelo[
    (df_rucua_modelo["bey_count"] > 0)
    & df_rucua_modelo["price"].notna()
    & (df_rucua_modelo["price"] > 0)
    & df_rucua_modelo["currency"].eq("MXN")
].copy()

df_pca_productos["_join_key"] = 1
ingresos_modelo["_join_key"] = 1

dataset_pca_input = (
    df_pca_productos
    .merge(
        ingresos_modelo,
        on="_join_key"
    )
    .drop(
        columns="_join_key"
    )
)

dataset_pca_input["price_income_pct"] = (
    dataset_pca_input["price"]
    /
    dataset_pca_input[
        "ingreso_personal_mensual_mxn"
    ]
    * 100
)

dataset_pca_input["price_per_bey"] = (
    dataset_pca_input["price"]
    /
    dataset_pca_input["bey_count"]
)

print(
    "Productos base:",
    len(df_pca_productos)
)

print(
    "Escenarios de ingreso:",
    len(ingresos_modelo)
)

print(
    "Observaciones finales:",
    len(dataset_pca_input)
)

Productos base: 37
Escenarios de ingreso: 11
Observaciones finales: 407


### 7.1 Variables candidatas para PCA

Esta lista **no ejecuta** PCA. Solo deja documentadas las variables numéricas que pueden evaluarse en el siguiente notebook.

En el notebook de PCA se decidirá si conviene retirar variables muy correlacionadas o redundantes antes del ajuste final.

In [32]:
pca_candidate_features = [
    "price",

    "bey_count",
    "blade_count",
    "ratchet_count",
    "bit_count",
    "launcher_count",
    "stadium_count",

    "launcher_included",
    "stadium_included",
    "in_stock",

    "package_usage_mean",
    "package_usage_max",

    "package_top_cuts_mean",
    "package_top_cuts_max",

    "package_first_rate_mean",
    "package_first_rate_max",

    "package_win_share_mean",
    "package_win_share_max",

    "package_match_coverage",
    "package_meta_coverage",

    "ingreso_personal_mensual_mxn",
    "price_income_pct",
    "price_per_bey"
]

columnas_existentes = [
    c
    for c in pca_candidate_features
    if c in dataset_pca_input.columns
]

print("Variables candidatas:")
for col in columnas_existentes:
    print("-", col)

print("\nNulos en variables candidatas:")
print(
    dataset_pca_input[
        columnas_existentes
    ].isnull().sum()
)

Variables candidatas:
- price
- bey_count
- blade_count
- ratchet_count
- bit_count
- launcher_count
- stadium_count
- launcher_included
- stadium_included
- in_stock
- package_usage_mean
- package_usage_max
- package_top_cuts_mean
- package_top_cuts_max
- package_first_rate_mean
- package_first_rate_max
- package_win_share_mean
- package_win_share_max
- package_match_coverage
- package_meta_coverage
- ingreso_personal_mensual_mxn
- price_income_pct
- price_per_bey

Nulos en variables candidatas:
price                           0
bey_count                       0
blade_count                     0
ratchet_count                   0
bit_count                       0
launcher_count                  0
stadium_count                   0
launcher_included               0
stadium_included                0
in_stock                        0
package_usage_mean              0
package_usage_max               0
package_top_cuts_mean           0
package_top_cuts_max            0
package_first_rate_mea

### 7.2 Exportación final

`beyblade_pca_input.csv` es el archivo principal para el siguiente notebook.

También se guarda un JSON con la lista de variables candidatas para que el notebook de PCA pueda reutilizarla.

In [33]:
dataset_pca_input.to_csv(
    "beyblade_pca_input.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    "pca_candidate_features.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        columnas_existentes,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Archivos finales creados:")
print("- beywatch_meta_raw.csv")
print("- beywatch_meta_clean.csv")
print("- rucua_products_raw.csv")
print("- rucua_products_clean.csv")
print("- rucua_package_contents.csv")
print("- rucua_contents_with_meta.csv")
print("- rucua_model_dataset.csv")
print("- beyblade_pca_input.csv")
print("- pca_candidate_features.json")

Archivos finales creados:
- beywatch_meta_raw.csv
- beywatch_meta_clean.csv
- rucua_products_raw.csv
- rucua_products_clean.csv
- rucua_package_contents.csv
- rucua_contents_with_meta.csv
- rucua_model_dataset.csv
- beyblade_pca_input.csv
- pca_candidate_features.json


# 8. Checklist final

Antes de abrir el notebook de PCA, revisa:

- `problemas_composicion` debe estar vacío.
- los errores de scraping restantes deben ser 0 o estar documentados;
- `beyblade_pca_input.csv` debe tener más filas que productos, porque cada producto se cruza con varios escenarios de ingreso;
- los `NaN` de meta no deben haberse convertido automáticamente en 0;
- `package_meta_coverage` permite saber cuánta información competitiva real respalda cada producto.

El siguiente notebook puede empezar directamente leyendo:

```python
import pandas as pd
import json

df = pd.read_csv("beyblade_pca_input.csv")

with open("pca_candidate_features.json", encoding="utf-8") as f:
    features = json.load(f)
```

A partir de ahí se realizará:

`EDA → selección de variables → StandardScaler → PCA → varianza explicada → loadings → interpretación`.